In [ ]:
!pip install yt-dlp openai-whisper faiss-cpu sentence-transformers groq gradio

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your_api_key_here"

In [ ]:
import yt_dlp

def download_audio(url):
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'audio.%(ext)s',
        'quiet': True
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    
    return "audio.webm"

In [ ]:
import whisper

whisper_model = whisper.load_model("base")  # use "small" if better quality needed

def transcribe_audio(file_path):
    result = whisper_model.transcribe(file_path)
    return result["text"]

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def create_vector_store(chunks):
    embeddings = embed_model.encode(chunks)
    
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))
    
    return index, embeddings

In [ ]:
def retrieve(query, chunks, index, k=3):
    query_embedding = embed_model.encode([query])
    
    distances, indices = index.search(np.array(query_embedding), k)
    
    return [chunks[i] for i in indices[0]]

In [ ]:
from groq import Groq

client = Groq()

def generate_answer(query, context):
    prompt = f"""
You are a helpful assistant.

Answer ONLY from the provided context.
If the answer is not in the context, say "Not found in video".

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="llama3-70b-8192",  # free + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    
    return response.choices[0].message.content

In [ ]:
chunks = None
index = None

def process_video(url):
    global chunks, index
    
    audio_path = download_audio(url)
    transcript = transcribe_audio(audio_path)
    
    chunks = chunk_text(transcript)
    index, _ = create_vector_store(chunks)
    
    return "✅ Video processed successfully!"

In [ ]:
def ask_question(query):
    global chunks, index
    
    if chunks is None:
        return "⚠️ Please load a video first."
    
    retrieved_chunks = retrieve(query, chunks, index)
    context = "\n".join(retrieved_chunks)
    
    answer = generate_answer(query, context)
    return answer

In [ ]:
import gradio as gr

with gr.Blocks() as app:
    gr.Markdown("# 🎥 YouTube RAG Q&A (Groq + Whisper)")
    
    with gr.Tab("Load Video"):
        url_input = gr.Textbox(label="YouTube URL")
        load_btn = gr.Button("Process Video")
        load_output = gr.Textbox()
    
    with gr.Tab("Ask Questions"):
        query_input = gr.Textbox(label="Ask something about the video")
        ask_btn = gr.Button("Get Answer")
        answer_output = gr.Textbox()
    
    load_btn.click(process_video, inputs=url_input, outputs=load_output)
    ask_btn.click(ask_question, inputs=query_input, outputs=answer_output)

app.launch(debug = True)